# Lab Exercise: Three Learning Paradigms on One Dataset
This notebook runs **Supervised**, **Starved Supervised**, **Semi-Supervised (Self-Training)**, and **Unsupervised (KMeans Clustering)** machine learning models on the exact same Breast Cancer dataset.

The only thing we vary is **how many labels the model is allowed to see during training**.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

warnings.filterwarnings("ignore")

# Load breast cancer dataset
data = load_breast_cancer()
X, y = data.data, data.target

print("THE DATA")
print(f"  samples (rows)       {X.shape[0]}")
print(f"  features (columns)   {X.shape[1]} — all numeric")
print(f"  classes              {data.target_names[0]}={np.sum(y == 0)}, "
      f"{data.target_names[1]}={np.sum(y == 1)}")
print(f"  label 1 means        \'{data.target_names[1]}'  <- sklearn\'s positive class")

# Split into 75% train, 25% test
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
print(f"  train {X_tr.shape[0]} rows, test {X_te.shape[0]} rows")

### Helper Pipeline function
To make things fair, all models will use standard scaling followed by standard logistic regression with a high iteration limit to ensure convergence.

In [ ]:
def pipe():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

### 1. Supervised Learning (All 426 Training Labels Visible)
We train the pipeline on the full training set using 100% of the target labels.

In [ ]:
full_model = pipe().fit(X_tr, y_tr)
full_accuracy = accuracy_score(y_te, full_model.predict(X_te))
print(f"Fully Supervised Accuracy: {full_accuracy:.3f}")

### 2 & 3. Starved Supervised vs. Semi-Supervised
We starve the model by showing it only **30 labels** out of 426, throwing away the rest (Starved Supervised).
Then, we run **Semi-Supervised (Self-Training)** on those same 30 labels *plus* the 396 unlabelled rows (marked with `-1`).

We run this experiment over **15 random seeds** to avoid reporting a single lucky split.

In [ ]:
N_LAB, SEEDS = 30, 15
starved, semi = [], []

for seed in range(SEEDS):
    # Random permutation of training indices
    idx = np.random.RandomState(seed).permutation(len(y_tr))
    lab, unlab = idx[:N_LAB], idx[N_LAB:]
    
    # Check we have at least one example of both classes in our sample
    if len(np.unique(y_tr[lab])) < 2:
        continue
        
    # Starved Supervised
    starved_model = pipe().fit(X_tr[lab], y_tr[lab])
    starved.append(accuracy_score(y_te, starved_model.predict(X_te)))
    
    # Semi-Supervised Self-Training
    y_semi = np.copy(y_tr)
    y_semi[unlab] = -1  # -1 marks label as unknown
    
    self_trainer = SelfTrainingClassifier(pipe())
    self_trainer.fit(X_tr, y_semi)
    semi.append(accuracy_score(y_te, self_trainer.predict(X_te)))

mean_starved = np.mean(starved)
mean_semi = np.mean(semi)
print(f"Starved Supervised (30 labels only) Mean Accuracy: {mean_starved:.3f}")
print(f"Semi-Supervised (30 labels + 396 unlabelled) Mean Accuracy: {mean_semi:.3f}")

### 4. Unsupervised Learning (Zero Labels Visible)
We run KMeans with `n_clusters=2` on the scaled features without using target labels at all.
We check agreement by trying both cluster assignments, matching the cluster to the classes.

In [ ]:
Xs = StandardScaler().fit_transform(X_tr)
km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(Xs)

# Check both possible mappings of cluster IDs (0/1) to class IDs (0/1)
unsupervised_accuracy = max(accuracy_score(y_tr, km.labels_), accuracy_score(y_tr, 1 - km.labels_))
ari = adjusted_rand_score(y_tr, km.labels_)

print(f"Unsupervised (KMeans) Training Label Recovery Accuracy: {unsupervised_accuracy:.3f}")
print(f"KMeans Adjusted Rand Index (corrected for chance): {ari:.3f}")

### Summary Comparison
Let's display the final comparative results.

In [ ]:
results = pd.DataFrame({
    "Paradigm": ["Supervised (Full)", "Supervised (Starved)", "Semi-Supervised", "Unsupervised (KMeans)"],
    "Labels Used": [len(y_tr), N_LAB, N_LAB, 0],
    "Unlabelled Rows Used": [0, "discarded", len(y_tr) - N_LAB, len(y_tr)],
    "Accuracy": [full_accuracy, mean_starved, mean_semi, unsupervised_accuracy]
})
print(results.to_string(index=False))

### Lab Experiments (Things to Try)

#### Experiment 1: Plot the Learning Curve
1. Change `N_LAB` inside the cell above to values like `5`, `10`, `50`, `100`, `200`.
2. Observe what happens to the accuracy of Starved Supervised vs. Semi-Supervised.
3. Notice how the accuracy curve rises steeply then flattens out, showing you the exact point of diminishing returns for human labelling efforts!

#### Experiment 2: The Multi-Cluster Mirage
1. Change `n_clusters` in KMeans to 3, and then 4.
2. Check if scikit-learn prints any error. (It won't!). It will happily segment your two-class data into four groups. This highlights why unsupervised learning is hard to evaluate objectively—there is no baseline accuracy.